# Project 02 – FAQBot for Douglas College FAQs

**Group members**  
- Dundee Adriatico – 300393449 
- Full Name 2 – Student ID  
...

## References
- Douglas College FAQ web pages (project handout)
- JPMorgan FAQ Bot tutorial using Elasticsearch and embeddings
- Datacamp Google File Search / Gemini RAG tutorial

## Planning

We will:
1. Collect Douglas College FAQ data (questions and answers) into a CSV or JSON file.
2. Build a simple keyword-based FAQBot using Elasticsearch: index each FAQ, search with user questions.
3. Evaluate with 5 original questions per group member using Precision@1.
4. Improve the bot with question embeddings using `en_core_web_lg` and Elasticsearch dense vectors.
5. Build a RAG-style FAQBot using Gemini + File Search following the Datacamp tutorial, then test with the same questions. 

In [ ]:
# Part B – imports

from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk
import pandas as pd
import numpy as np

# Load FAQ data from a prepared CSV file with columns: id, question, answer
faq_df = pd.read_csv("douglas_faqs.csv")  # you create this file beforehand
faq_df.head()


In [ ]:
# Connect to local Elasticsearch
es = Elasticsearch("http://localhost:9200")

INDEX_B = "faq_keyword"

# Delete and create index
if es.indices.exists(index=INDEX_B):
    es.indices.delete(index=INDEX_B)

es.indices.create(
    index=INDEX_B,
    body={
        "mappings": {
            "properties": {
                "question": {"type": "text"},
                "answer": {"type": "text"}
            }
        }
    }
)

# Bulk index documents
actions = [
    {
        "_index": INDEX_B,
        "_id": int(row["id"]),
        "_source": {
            "question": row["question"],
            "answer": row["answer"]
        },
    }
    for _, row in faq_df.iterrows()
]

bulk(es, actions)


In [ ]:
# Simple keyword search using match on question field

def search_keyword(query, k=5):
    res = es.search(
        index=INDEX_B,
        body={
            "size": k,
            "query": {
                "multi_match": {
                    "query": query,
                    "fields": ["question", "answer"]
                }
            }
        }
    )
    hits = res["hits"]["hits"]
    return [
        {
            "score": h["_score"],
            "question": h["_source"]["question"],
            "answer": h["_source"]["answer"],
        }
        for h in hits
    ]

# Quick test
search_keyword("How do I pay my fees?", k=3)


### Test questions (Member 1)

1. How can I drop a course after the deadline?
2. When are tuition fees due for winter term?
3. How do I book an appointment with an academic advisor?
4. What is the policy for online exam proctoring?
5. How do I request a transcript to be sent to another school?


In [ ]:
# Replace with all questions from all members
test_questions_B = [
    "How can I drop a course after the deadline?",
    "When are tuition fees due for winter term?",
    "How do I book an appointment with an academic advisor?",
    "What is the policy for online exam proctoring?",
    "How do I request a transcript to be sent to another school?",
    # ... add the rest for other members
]

# For simplicity, store relevance judgments manually:
# 1 means first result is relevant, 0 means not relevant.
precision_at_1_labels_B = []

for q in test_questions_B:
    results = search_keyword(q, k=1)
    print("QUERY:", q)
    if results:
        print("TOP ANSWER:", results[0]["answer"])
    # After running, manually inspect and enter 1 or 0.
    label = int(input("Is this relevant? (1/0): "))
    precision_at_1_labels_B.append(label)

avg_p1_B = np.mean(precision_at_1_labels_B)
print("Average Precision@1 for Part B:", avg_p1_B)


In [ ]:
# pip install spacy
# python -m spacy download en_core_web_lg

import spacy
nlp = spacy.load("en_core_web_lg")


In [ ]:
INDEX_C = "faq_embed"

if es.indices.exists(index=INDEX_C):
    es.indices.delete(index=INDEX_C)

EMBED_DIM = 300  # en_core_web_lg vector size

es.indices.create(
    index=INDEX_C,
    body={
        "mappings": {
            "properties": {
                "question": {"type": "text"},
                "answer": {"type": "text"},
                "question_vector": {
                    "type": "dense_vector",
                    "dims": EMBED_DIM
                }
            }
        }
    }
)


In [ ]:
def embed(text):
    return nlp(text).vector.tolist()

actions_c = []
for _, row in faq_df.iterrows():
    actions_c.append(
        {
            "_index": INDEX_C,
            "_id": int(row["id"]),
            "_source": {
                "question": row["question"],
                "answer": row["answer"],
                "question_vector": embed(row["question"])
            },
        }
    )

bulk(es, actions_c)


In [ ]:
def search_embedding(query, k=5):
    q_vec = embed(query)
    res = es.search(
        index=INDEX_C,
        body={
            "size": k,
            "query": {
                "script_score": {
                    "query": {"match_all": {}},
                    "script": {
                        "source": "cosineSimilarity(params.q_vec, 'question_vector') + 1.0",
                        "params": {"q_vec": q_vec},
                    },
                }
            },
        },
    )
    hits = res["hits"]["hits"]
    return [
        {
            "score": h["_score"],
            "question": h["_source"]["question"],
            "answer": h["_source"]["answer"],
        }
        for h in hits
    ]

# quick test
search_embedding("How do I pay my tuition fees?", k=3)


In [ ]:
test_questions_C = test_questions_B  # same questions

precision_at_1_labels_C = []

for q in test_questions_C:
    results = search_embedding(q, k=1)
    print("QUERY:", q)
    if results:
        print("TOP ANSWER:", results[0]["answer"])
    label = int(input("Is this relevant? (1/0): "))
    precision_at_1_labels_C.append(label)

avg_p1_C = np.mean(precision_at_1_labels_C)
print("Average Precision@1 for Part C:", avg_p1_C)


In [ ]:
import os
from google import genai

os.environ["GOOGLE_API_KEY"] = "YOUR_KEY_HERE"  # better: use .env

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

FILE_SEARCH_STORE_NAME = "douglas_faq_store"

# Use helper functions from the tutorial to create/get the store and upload douglas_faqs.csv
# (keep this exactly as shown in tutorial but remove extra fancy parts). [web:10]


In [ ]:
def ask_gemini_faq(question):
    prompt = (
        "You are an FAQ assistant for Douglas College.\n"
        "Answer only using the uploaded FAQ documents. "
        "If you are not sure, say you do not know.\n\n"
        f"Question: {question}"
    )

    response = client.models.generate_content(
        model="gemini-2.0-flash",  # or model from tutorial
        contents=prompt,
        config={
            "tools": [
                {
                    "file_search": {
                        "file_search_store_names": [FILE_SEARCH_STORE_NAME]
                    }
                }
            ]
        }
    )
    return response.text

# quick test
print(ask_gemini_faq("When are tuition fees due?"))


In [ ]:
test_questions_D = test_questions_B

precision_at_1_labels_D = []

for q in test_questions_D:
    answer = ask_gemini_faq(q)
    print("QUERY:", q)
    print("ANSWER:", answer)
    label = int(input("Is this relevant? (1/0): "))
    precision_at_1_labels_D.append(label)

avg_p1_D = np.mean(precision_at_1_labels_D)
print("Average Precision@1 for Part D:", avg_p1_D)


## Conclusion

- Part B (keyword): Average Precision@1 = X.XX
- Part C (embedding): Average Precision@1 = Y.YY
- Part D (Gemini RAG): Average Precision@1 = Z.ZZ

The keyword-only Elasticsearch bot works but often fails when the question wording is different from the FAQ titles. Embedding-based search improves semantic matching and gives better average Precision@1. Gemini RAG, using the same FAQ documents, usually provides the most natural answers and highest Precision@1, but depends on external API and may be slower. [web:6][web:10]
